## Single Layer DNN Models

In [2]:
import os
from pathlib import Path

import kagglehub
import tensorflow as tf

# set current data path
data_path = os.getcwd()
# import kaggle token
token = Path("kaggletoken.txt").read_text().strip()
os.environ["KAGGLE_API_TOKEN"] = token

# define kaggle download
download_path = kagglehub.dataset_download(
    "vincentvaseghi/us-cities-housing-market-data",
    path="city_market_tracker.tsv000",
    output_dir=data_path + "/data",
)

# check rocm devices
devices = tf.config.list_physical_devices("GPU")
print("GPUs found:", devices)

/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0000 00:00:1773936746.085909      27 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773936746.085976      27 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773936746.085979      27 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773936746.085981      27 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
2026-03-19 12:12:26.095628: I tensorflow/core/platform/cpu_f

GPUs found: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Import into pandas

In [3]:
import pandas as pd

# load the tsv as a csv but with tab seperators
df = pd.read_csv(data_path + "/data/city_market_tracker.tsv000", sep="\t")

# print head of dataset
print(df.head())
df.columns.tolist()

  PERIOD_BEGIN  PERIOD_END  PERIOD_DURATION REGION_TYPE  REGION_TYPE_ID  \
0   2021-08-01  2021-08-31               30       place               6   
1   2022-03-01  2022-03-31               30       place               6   
2   2016-09-01  2016-09-30               30       place               6   
3   2025-12-01  2025-12-31               30       place               6   
4   2018-03-01  2018-03-31               30       place               6   

   TABLE_ID  IS_SEASONALLY_ADJUSTED            REGION          CITY  \
0     14689                   False   St. Augusta, MN   St. Augusta   
1     30390                   False       Newfane, NY       Newfane   
2     18083                   False  Simpsonville, KY  Simpsonville   
3     11261                   False         Maize, KS         Maize   
4     30755                   False    Loganville, GA    Loganville   

       STATE  ... SOLD_ABOVE_LIST_YOY PRICE_DROPS  PRICE_DROPS_MOM  \
0  Minnesota  ...            0.354167    0.272727   

['PERIOD_BEGIN',
 'PERIOD_END',
 'PERIOD_DURATION',
 'REGION_TYPE',
 'REGION_TYPE_ID',
 'TABLE_ID',
 'IS_SEASONALLY_ADJUSTED',
 'REGION',
 'CITY',
 'STATE',
 'STATE_CODE',
 'PROPERTY_TYPE',
 'PROPERTY_TYPE_ID',
 'MEDIAN_SALE_PRICE',
 'MEDIAN_SALE_PRICE_MOM',
 'MEDIAN_SALE_PRICE_YOY',
 'MEDIAN_LIST_PRICE',
 'MEDIAN_LIST_PRICE_MOM',
 'MEDIAN_LIST_PRICE_YOY',
 'MEDIAN_PPSF',
 'MEDIAN_PPSF_MOM',
 'MEDIAN_PPSF_YOY',
 'MEDIAN_LIST_PPSF',
 'MEDIAN_LIST_PPSF_MOM',
 'MEDIAN_LIST_PPSF_YOY',
 'HOMES_SOLD',
 'HOMES_SOLD_MOM',
 'HOMES_SOLD_YOY',
 'PENDING_SALES',
 'PENDING_SALES_MOM',
 'PENDING_SALES_YOY',
 'NEW_LISTINGS',
 'NEW_LISTINGS_MOM',
 'NEW_LISTINGS_YOY',
 'INVENTORY',
 'INVENTORY_MOM',
 'INVENTORY_YOY',
 'MONTHS_OF_SUPPLY',
 'MONTHS_OF_SUPPLY_MOM',
 'MONTHS_OF_SUPPLY_YOY',
 'MEDIAN_DOM',
 'MEDIAN_DOM_MOM',
 'MEDIAN_DOM_YOY',
 'AVG_SALE_TO_LIST',
 'AVG_SALE_TO_LIST_MOM',
 'AVG_SALE_TO_LIST_YOY',
 'SOLD_ABOVE_LIST',
 'SOLD_ABOVE_LIST_MOM',
 'SOLD_ABOVE_LIST_YOY',
 'PRICE_DROPS',
 'PRICE_DRO

## Clean the data

In [19]:
import numpy as np
import sklearn as sk

# drop na data
df = df.dropna()

# convert period start to real date time
df["PERIOD_BEGIN"] = pd.to_datetime(df["PERIOD_BEGIN"])
# sort
df = df.sort_values("PERIOD_BEGIN")
# add new columns for dates
df["MONTH"] = pd.to_datetime(df["PERIOD_BEGIN"]).dt.month
df["YEAR"] = pd.to_datetime(df["PERIOD_BEGIN"]).dt.year

# get sine of season to see seaonal affects
df["MONTH_SIN"] = np.sin(2 * np.pi * df["MONTH"] / 12)
df["MONTH_COS"] = np.cos(2 * np.pi * df["MONTH"] / 12)

# be able to learn trends from previous months from regions
df["PRICE_LAG_1"] = df.groupby("REGION")["MEDIAN_SALE_PRICE"].shift(1)
df["PRICE_LAG_3"] = df.groupby("REGION")["MEDIAN_SALE_PRICE"].shift(3)
# drop what we arert going to use or compare
df = df.drop(columns=["PERIOD_END", "PERIOD_DURATION", "TABLE_ID"])

# we need to convert string categories to categories with an int value assigned

# define colums to be encoded
cols = df.select_dtypes(include=["object", "str"]).columns

oe = sk.preprocessing.OrdinalEncoder()
df[cols] = oe.fit_transform(df[cols])

KeyError: "['PERIOD_END', 'PERIOD_DURATION', 'TABLE_ID'] not found in axis"

## Pre Process before Training

In [20]:
# scale the data
scaled_data = pd.DataFrame(
    sk.preprocessing.MinMaxScaler().fit_transform(df), columns=df.columns
)

# define x and y data
x_data =[
    "STATE",
    "CITY",
    "PROPERTY_TYPE",
    "INVENTORY",
    "REGION",
    "HOMES_SOLD",
    "MEDIAN_DOM",
    "AVG_SALE_TO_LIST",
    "PRICE_LAG_1",
    "PRICE_LAG_3",
]
# predicitng median sale price and median price per square foot
y_data = ["MEDIAN_SALE_PRICE", "MEDIAN_PPSF"]

# pick test data of last three years
cutoff = df["PERIOD_BEGIN"].max() - pd.DateOffset(years=3)
cutoff_val  = cutoff_test - pd.DateOffset(years=2)

# copy to new dataset
train_df = df[df["PERIOD_BEGIN"] < cutoff_val].copy()
val_df = cutoff - pd.DataOffset(years=2)
test_df = df[df["PERIOD_BEGIN"] >= cutoff].copy()

# x and y train and test
x_test = test_df[x_data]
y_test = test_df[y_data]

x_train = train_df[x_data]
y_train = train_df[y_data]

x_val = val_df[x_data]
y_val = val_df[y_data]

DTypePromotionError: The DType <class 'numpy.dtypes.DateTime64DType'> could not be promoted by <class 'numpy.dtypes.Float64DType'>. This means that no common DType exists for the given inputs. For example they cannot be stored in a single array unless the dtype is `object`. The full list of DTypes is: (<class 'numpy.dtypes.DateTime64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.BoolDType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'

## Scoring Method

In [ ]:
import math

from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score


def scores(y_true, y_pred):
    return {
        "rmse": math.sqrt(mean_squared_error(y_true, y_pred)),
        "R2_score": r2_score(y_true, y_pred),
        "R": np.corrcoef(y_true, y_pred)[0, 1],
    }

## Build the Model

In [ ]:
# define neuron amounts and epoch amounts
neuron_options = [10, 20, 50, 100, 200, 300, 400, 500]
epoch_options = [10, 20, 50, 100, 200, 300, 400, 500]

# results array
results = []
# results for best epochs
best_results = []


def build_model(neurons, input_features, learning_rate=0.001):
    model = tf.Sequential()
    model.add(tf.Dense(neurons, input_dim=input_features, activation='relu'))
    model.add(tf.Dense(1, activation='linear'))

    model.compile(
        loss='mean_squared_error',
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate)
    )

    return model



## Run the model

In [ ]:
def run_one_model(neurons, epochs, learning_rate=0.001, batch_size=32):
    model = build_model(
        neurons=neurons,
        input_features=x_train.shape[1],
        learning_rate=learning_rate
    )

    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=0
    )

    y_pred = model.predict(x_val, verbose=0).ravel()
    model_scores = scores(y_val, y_pred)

    return model, history, model_scores

## Run Method

In [ ]:
# Run a single model
def run_one_model(neurons, epochs, learning_rate=0.001, batch_size=32):
    model = build_model(
        neurons=neurons,
        input_features=x_train.shape[1],
        learning_rate=learning_rate
    )

    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=0
    )

    y_val_pred = model.predict(x_val, verbose=0).ravel()
    val_scores = scores(y_val, y_val_pred)

    return model, history, val_scores




def run_all_models(neuron_options, epoch_options):
    results = []
    best_model = None
    best_result = None
    best_rmse = float("inf")
    # go through and run all the models for each neuron and epoch amount
    for neurons in neuron_options:
        for epochs in epoch_options:
            model, history, val_scores = run_one_model(neurons, epochs)

            result = {
                "neurons": neurons,
                "epochs": epochs,
                "rmse": val_scores["rmse"],
                "R2_score": val_scores["R2_score"],
                "R": val_scores["R"],
                "model": model,
                "history": history
            }

            results.append(result)

            if val_scores["rmse"] < best_rmse:
                best_rmse = val_scores["rmse"]
                best_model = model
                best_result = result
    #store all the results ina  dataframe
    results_df = pd.DataFrame([
        {
            "neurons": r["neurons"],
            "epochs": r["epochs"],
            "rmse": r["rmse"],
            "R2_score": r["R2_score"],
            "R": r["R"]
        }
        for r in results
    ]).sort_values("rmse")

    return results, results_df, best_model, best_result

## Run the models